# Trial Conversion Analysis — Tidemill API

Trial-to-paid conversion rates, funnel analysis, and monthly trends from the Tidemill engine.

**API base:** `http://localhost:8000` (after `make seed && make dev`) or set `TIDEMILL_API` env var

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import requests

warnings.filterwarnings("ignore", "Unverified HTTPS")

API = os.environ.get("TIDEMILL_API", "http://localhost:8000")

START, END = "2025-09-01", "2026-03-31"

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.3})


def api_get(path, **params):
    r = requests.get(f"{API}{path}", params=params)
    r.raise_for_status()
    return r.json()

## 1. Overall Trial Conversion Rate

In [ ]:
rate = api_get("/api/metrics/trials", start=START, end=END)
if rate is not None:
    print(f"Trial conversion rate: {rate * 100:.1f}%")
else:
    print("Trial conversion rate: N/A (no trial starts in period)")

## 2. Trial Funnel

Breakdown of trial outcomes: started, converted to paid, and expired.

In [ ]:
funnel = api_get("/api/metrics/trials/funnel", start=START, end=END)

print(f"Trials started:    {funnel['started']}")
print(f"Trials converted:  {funnel['converted']}")
print(f"Trials expired:    {funnel['expired']}")
if funnel["conversion_rate"] is not None:
    print(f"Conversion rate:   {funnel['conversion_rate'] * 100:.1f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Funnel bar chart
labels = ["Started", "Converted", "Expired"]
values = [funnel["started"], funnel["converted"], funnel["expired"]]
colors = ["#3498db", "#2ecc71", "#e74c3c"]
bars = ax1.bar(labels, values, color=colors)
ax1.set_title("Trial Funnel")
ax1.set_ylabel("Count")
for bar, val in zip(bars, values, strict=True):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        str(val),
        ha="center",
        fontweight="bold",
        fontsize=12,
    )

# Conversion pie chart
if funnel["started"] > 0:
    converted = funnel["converted"]
    not_converted = funnel["started"] - converted
    ax2.pie(
        [converted, not_converted],
        labels=["Converted", "Not converted"],
        autopct="%1.0f%%",
        colors=["#2ecc71", "#e74c3c"],
        startangle=90,
        explode=[0.05, 0],
    )
    ax2.set_title("Trial Conversion")

plt.tight_layout()
plt.show()

## 3. Monthly Conversion Rate Timeline

Track how trial conversion rate changes over time.

In [ ]:
series = api_get("/api/metrics/trials/series", start=START, end=END, interval="month")

if series:
    df_ts = pd.DataFrame(series)

    print("Monthly trial metrics:")
    for _, row in df_ts.iterrows():
        rate_s = f"{row.conversion_rate * 100:.0f}%" if row.conversion_rate is not None else "N/A"
        print(
            f"  {row.period}:  started={row.started}  converted={row.converted}"
            f"  expired={row.expired}  rate={rate_s}"
        )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9))

    # Stacked bar: started / converted / expired per month
    x = range(len(df_ts))
    ax1.bar(x, df_ts.converted, color="#2ecc71", label="Converted")
    ax1.bar(x, df_ts.expired, bottom=df_ts.converted, color="#e74c3c", label="Expired")
    pending = df_ts.started - df_ts.converted - df_ts.expired
    pending = pending.clip(lower=0)
    ax1.bar(x, pending, bottom=df_ts.converted + df_ts.expired, color="#95a5a6", label="Pending")

    ax1.set_xticks(x)
    period_labels = [str(p)[:7] if len(str(p)) > 7 else str(p) for p in df_ts.period]
    ax1.set_xticklabels(period_labels, rotation=45)
    ax1.set_title("Monthly Trial Outcomes")
    ax1.set_ylabel("Count")
    ax1.legend()

    # Conversion rate line
    rates = [r * 100 if r is not None else float("nan") for r in df_ts.conversion_rate]
    ax2.plot(x, rates, "o-", color="#2ecc71", linewidth=2.5, markersize=8)
    ax2.fill_between(x, rates, alpha=0.1, color="#2ecc71")
    ax2.axhline(50, color="#95a5a6", linewidth=1, linestyle=":")

    for i, r in enumerate(rates):
        if not pd.isna(r):
            ax2.annotate(
                f"{r:.0f}%",
                (i, r),
                textcoords="offset points",
                xytext=(0, 12),
                ha="center",
                fontweight="bold",
            )

    ax2.set_xticks(x)
    ax2.set_xticklabels(period_labels, rotation=45)
    ax2.set_title("Monthly Trial Conversion Rate")
    ax2.set_ylabel("Conversion Rate (%)")
    ax2.set_ylim(0, 105)

    plt.tight_layout()
    plt.show()
else:
    print("No trial series data available.")